Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = ""
COLLABORATORS = ""

---

# Assignment 2: Sector Diversification and Portfolio Risk

**Created by: Pengfei Zhao**  
**© 2025 Investment Practice Course**

**Learning Objectives:**
- Understand the importance of sector diversification in portfolio construction
- Compare efficient frontiers between single-sector and multi-sector portfolios
- Analyze risk decomposition to understand correlation effects
- Demonstrate why "diversification" within a single sector provides limited risk reduction
- Apply portfolio optimization techniques to real stock data

**Key Concept:**
> **"Don't put all your eggs in one basket... and make sure your baskets are in different industries!"**
> 
> Holding multiple stocks from the same sector (e.g., all tech stocks) provides much less diversification benefit than holding stocks from different sectors. This is because stocks in the same sector tend to move together due to common risk factors.

---

## Mathematical Foundation

### Portfolio Variance and Correlation

The portfolio variance for n assets is:

$$\sigma_p^2 = \sum_{i=1}^{n} \sum_{j=1}^{n} w_i w_j \sigma_i \sigma_j \rho_{ij}$$

Where:
- $w_i$ = weight of asset i
- $\sigma_i$ = volatility (standard deviation) of asset i
- $\rho_{ij}$ = correlation coefficient between assets i and j

### Why Sector Diversification Matters

**High Correlation Problem**: Stocks in the same sector typically have high correlations ($\rho \approx 0.7-0.9$) because they:
- Face similar market conditions
- Are affected by the same industry trends
- Respond to common regulatory changes
- Share similar customer bases and supply chains

**Diversification Benefit**: The diversification benefit is measured by:

$$DB = \frac{\text{Weighted Avg Volatility} - \text{Portfolio Volatility}}{\text{Weighted Avg Volatility}} \times 100\%$$

Higher diversification benefits occur when correlations are lower!

---

## Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('default')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
sns.set_palette("husl")
print("="*50)
print("✅ Libraries imported successfully")

### Load Portfolio Data

We will compare two portfolios:

**Portfolio 1 - Tech Sector Only:**
- AAPL (Apple)
- GOOGL (Alphabet/Google)
- MSFT (Microsoft)
- AMZN (Amazon)
- TSLA (Tesla)

**Portfolio 2 - Diversified Sectors:**
- AAPL (Technology)
- JPM (Financial Services)
- JNJ (Healthcare)
- WMT (Consumer Cyclical)
- PG (Consumer Defensive)

In [ ]:
# Load portfolio data
portfolio1_returns = pd.read_csv('portfolio1_tech_sector_returns.csv', index_col=0, parse_dates=True)
portfolio2_returns = pd.read_csv('portfolio2_diversified_sectors_returns.csv', index_col=0, parse_dates=True)

print("\n📈 Portfolio 1 - Tech Sector (All Technology Stocks):")
print(f"   Stocks: {list(portfolio1_returns.columns)}")
print(f"   Data points: {len(portfolio1_returns)}")
print(f"   Date range: {portfolio1_returns.index[0].date()} to {portfolio1_returns.index[-1].date()}")

print("\n📊 Portfolio 2 - Diversified Sectors:")
print(f"   Stocks: {list(portfolio2_returns.columns)}")
print(f"   Data points: {len(portfolio2_returns)}")
print(f"   Sectors: Technology, Finance, Healthcare, Consumer Cyclical, Consumer Defensive")

---

## Exercise 1: Calculate Basic Portfolio Statistics

For each portfolio, calculate:
1. Annualized mean returns (multiply daily returns by 252)
2. Annualized volatilities (multiply daily std by √252)
3. Correlation matrices
4. Covariance matrices (annualized)

Compare the correlation structures between the two portfolios.

In [ ]:
# Portfolio 1: Tech Sector
p1_annual_returns = portfolio1_returns.mean() * 252
p1_annual_volatilities = portfolio1_returns.std() * np.sqrt(252)
p1_correlation_matrix = portfolio1_returns.corr()
p1_covariance_matrix = portfolio1_returns.cov() * 252

print("\n📊 PORTFOLIO 1 - TECH SECTOR STATISTICS:")
print("="*50)
print("\nAnnualized Returns:")
print(p1_annual_returns.apply(lambda x: f"{x:.2%}"))
print("\nAnnualized Volatilities:")
print(p1_annual_volatilities.apply(lambda x: f"{x:.2%}"))
print(f"\nAverage Correlation: {p1_correlation_matrix.values[np.triu_indices_from(p1_correlation_matrix.values, k=1)].mean():.3f}")

# Portfolio 2: Diversified
p2_annual_returns = portfolio2_returns.mean() * 252
p2_annual_volatilities = portfolio2_returns.std() * np.sqrt(252)
p2_correlation_matrix = portfolio2_returns.corr()
p2_covariance_matrix = portfolio2_returns.cov() * 252

print("\n\n📊 PORTFOLIO 2 - DIVERSIFIED SECTORS STATISTICS:")
print("="*50)
print("\nAnnualized Returns:")
print(p2_annual_returns.apply(lambda x: f"{x:.2%}"))
print("\nAnnualized Volatilities:")
print(p2_annual_volatilities.apply(lambda x: f"{x:.2%}"))
print(f"\nAverage Correlation: {p2_correlation_matrix.values[np.triu_indices_from(p2_correlation_matrix.values, k=1)].mean():.3f}")

### Visualize Correlation Matrices

**Question**: Which portfolio has higher average correlations? What does this imply for diversification potential?

In [ ]:
# Visualize correlation matrices side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Portfolio 1: Tech Sector
sns.heatmap(p1_correlation_matrix, annot=True, fmt='.3f', cmap='RdYlBu_r', 
            center=0, square=True, linewidths=0.5, 
            cbar_kws={"shrink": .8}, vmin=-1, vmax=1, ax=ax1)
ax1.set_title('Portfolio 1: Tech Sector\nCorrelation Matrix', fontsize=14, fontweight='bold')

# Portfolio 2: Diversified
sns.heatmap(p2_correlation_matrix, annot=True, fmt='.3f', cmap='RdYlBu_r', 
            center=0, square=True, linewidths=0.5, 
            cbar_kws={"shrink": .8}, vmin=-1, vmax=1, ax=ax2)
ax2.set_title('Portfolio 2: Diversified Sectors\nCorrelation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Key Observation:")
print(f"   Tech Sector Avg Correlation: {p1_correlation_matrix.values[np.triu_indices_from(p1_correlation_matrix.values, k=1)].mean():.3f}")
print(f"   Diversified Avg Correlation: {p2_correlation_matrix.values[np.triu_indices_from(p2_correlation_matrix.values, k=1)].mean():.3f}")
print("\n   ⚠️  Tech stocks are highly correlated - they move together!")
print("   ✅ Diversified sectors have lower correlations - better diversification potential!")

---

## Exercise 2: Generate Efficient Frontiers

For each portfolio, generate the efficient frontier by:
1. Defining target returns from minimum to maximum
2. For each target return, minimize portfolio variance subject to:
   - Weights sum to 1
   - Portfolio return equals target return
   - No short selling (weights ≥ 0)
3. Plot both efficient frontiers on the same figure

**Expected Result**: The diversified portfolio should have a more favorable efficient frontier (lower risk for same return).

In [ ]:
def portfolio_volatility(weights, cov_matrix):
    """Calculate portfolio volatility"""
    return np.sqrt(np.dot(weights, np.dot(cov_matrix, weights)))

def portfolio_return(weights, expected_returns):
    """Calculate portfolio expected return"""
    return np.dot(weights, expected_returns)

def generate_efficient_frontier(expected_returns, cov_matrix, num_portfolios=30):
    """
    Generate efficient frontier points
    
    For each target return, find the portfolio with minimum variance
    subject to: weights sum to 1, return = target, weights >= 0
    """
    n_assets = len(expected_returns)
    
    # Define target returns range
    min_ret = expected_returns.min()
    max_ret = expected_returns.max()
    target_returns = np.linspace(min_ret, max_ret, num_portfolios)
    
    efficient_portfolios = []
    
    for target_return in target_returns:
        # Constraints
        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},  # Weights sum to 1
            {'type': 'eq', 'fun': lambda w, tr=target_return: portfolio_return(w, expected_returns) - tr}  # Return = target
        ]
        
        # Bounds: no short selling
        bounds = tuple((0, 1) for _ in range(n_assets))
        
        # Initial guess: equal weights
        initial_weights = np.array([1/n_assets] * n_assets)
        
        # Minimize variance
        result = minimize(
            fun=lambda w: portfolio_volatility(w, cov_matrix)**2,
            x0=initial_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'disp': False, 'ftol': 1e-9}
        )
        
        if result.success:
            vol = portfolio_volatility(result.x, cov_matrix)
            ret = portfolio_return(result.x, expected_returns)
            efficient_portfolios.append({
                'return': ret,
                'volatility': vol,
                'weights': result.x
            })
    
    return efficient_portfolios

print("\n🔄 Generating efficient frontiers...")
print("   This may take a moment...\n")

# Generate efficient frontiers
ef1 = generate_efficient_frontier(p1_annual_returns, p1_covariance_matrix)
print(f"✅ Portfolio 1 (Tech): {len(ef1)} efficient points generated")

ef2 = generate_efficient_frontier(p2_annual_returns, p2_covariance_matrix)
print(f"✅ Portfolio 2 (Diversified): {len(ef2)} efficient points generated")

### Plot Efficient Frontiers

Compare the two efficient frontiers side by side.

In [ ]:
# Extract data for plotting
ef1_vols = [p['volatility'] for p in ef1]
ef1_rets = [p['return'] for p in ef1]

ef2_vols = [p['volatility'] for p in ef2]
ef2_rets = [p['return'] for p in ef2]

# Create plot
fig, ax = plt.subplots(figsize=(14, 9))

# Plot efficient frontiers
ax.plot(ef1_vols, ef1_rets, 'r-', linewidth=3, label='Portfolio 1: Tech Sector', alpha=0.8)
ax.plot(ef2_vols, ef2_rets, 'b-', linewidth=3, label='Portfolio 2: Diversified Sectors', alpha=0.8)

# Plot individual assets for Portfolio 1
for i, stock in enumerate(portfolio1_returns.columns):
    ax.scatter(p1_annual_volatilities[i], p1_annual_returns[i], 
              s=100, alpha=0.6, color='red', marker='o', edgecolors='darkred', linewidth=1.5)
    ax.annotate(stock, (p1_annual_volatilities[i], p1_annual_returns[i]),
               xytext=(8, 8), textcoords='offset points', fontsize=10, 
               color='darkred', fontweight='bold')

# Plot individual assets for Portfolio 2
for i, stock in enumerate(portfolio2_returns.columns):
    ax.scatter(p2_annual_volatilities[i], p2_annual_returns[i], 
              s=100, alpha=0.6, color='blue', marker='s', edgecolors='darkblue', linewidth=1.5)
    ax.annotate(stock, (p2_annual_volatilities[i], p2_annual_returns[i]),
               xytext=(8, -12), textcoords='offset points', fontsize=10, 
               color='darkblue', fontweight='bold')

ax.set_xlabel('Portfolio Volatility (Annual)', fontsize=13, fontweight='bold')
ax.set_ylabel('Expected Return (Annual)', fontsize=13, fontweight='bold')
ax.set_title('Efficient Frontiers Comparison:\nTech Sector vs Diversified Sectors', 
            fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3, linestyle='--')

# Format axes as percentages
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, p: f'{y:.0%}'))

plt.tight_layout()
plt.show()

print("\n💡 Key Findings:")
print("   🔴 Red line (Tech Sector): Limited by high correlations between tech stocks")
print("   🔵 Blue line (Diversified): More favorable risk-return tradeoff")
print("\n   ✅ For the same level of return, the diversified portfolio achieves LOWER risk!")
print("   ✅ This demonstrates the power of cross-sector diversification!")

---

## Exercise 3: Risk Decomposition Analysis

Using the equal-weighted portfolios (20% in each stock), perform risk decomposition analysis:

1. Calculate marginal risk contributions for each asset
2. Calculate total risk contributions
3. Decompose variance into:
   - Individual asset variance contributions
   - Interaction (covariance) contributions
4. Compare the risk decomposition between the two portfolios

**Expected Finding**: The tech portfolio will have higher interaction variance (due to high correlations), while the diversified portfolio benefits from lower correlation effects.

In [ ]:
def risk_decomposition_analysis(weights, cov_matrix, asset_names):
    """
    Perform comprehensive risk decomposition analysis
    
    Returns:
    - Portfolio volatility
    - Component-wise analysis (marginal contributions, risk contributions, etc.)
    - Individual vs interaction variance breakdown
    """
    weights = np.array(weights)
    portfolio_variance = np.dot(weights, np.dot(cov_matrix, weights))
    portfolio_volatility = np.sqrt(portfolio_variance)
    
    # Marginal risk contributions (partial derivatives)
    marginal_contrib = np.dot(cov_matrix, weights) / portfolio_volatility
    
    # Risk contributions (weight × marginal contribution)
    risk_contrib = weights * marginal_contrib
    risk_contrib_pct = risk_contrib / risk_contrib.sum() * 100
    
    # Component risk analysis
    component_analysis = []
    
    for i, asset in enumerate(asset_names):
        # Individual variance contribution (diagonal element only)
        individual_variance = weights[i]**2 * cov_matrix[i, i]
        
        # Interaction effects (off-diagonal covariances)
        interaction_variance = 0
        for j in range(len(weights)):
            if i != j:
                interaction_variance += weights[i] * weights[j] * cov_matrix[i, j]
        
        component_analysis.append({
            'Asset': asset,
            'Weight %': weights[i] * 100,
            'Marginal Risk Contrib': marginal_contrib[i],
            'Risk Contribution': risk_contrib[i],
            'Risk Contrib %': risk_contrib_pct[i],
            'Individual Variance': individual_variance,
            'Interaction Variance': interaction_variance,
            'Risk Ratio': risk_contrib_pct[i] / (weights[i] * 100)  # Risk % / Weight %
        })
    
    return {
        'portfolio_volatility': portfolio_volatility,
        'portfolio_variance': portfolio_variance,
        'component_analysis': component_analysis,
        'total_individual_variance': sum([c['Individual Variance'] for c in component_analysis]),
        'total_interaction_variance': sum([c['Interaction Variance'] for c in component_analysis]) / 2  # Avoid double counting
    }

# Equal weights for both portfolios
n_assets = 5
equal_weights = np.array([1/n_assets] * n_assets)

print("\n📊 RISK DECOMPOSITION ANALYSIS")
print("="*50)
print(f"\nUsing equal-weighted portfolios: {equal_weights[0]:.1%} per asset\n")

# Portfolio 1 analysis
p1_risk_decomp = risk_decomposition_analysis(
    equal_weights, 
    p1_covariance_matrix.values, 
    portfolio1_returns.columns
)

print("\n🔴 PORTFOLIO 1 - TECH SECTOR:")
print(f"   Portfolio Volatility: {p1_risk_decomp['portfolio_volatility']:.2%}")
print("\n   Component Analysis:")
df1 = pd.DataFrame(p1_risk_decomp['component_analysis'])
print(df1[['Asset', 'Weight %', 'Risk Contrib %', 'Risk Ratio']].round(2).to_string(index=False))

# Variance decomposition
p1_individual_pct = p1_risk_decomp['total_individual_variance'] / p1_risk_decomp['portfolio_variance'] * 100
p1_interaction_pct = p1_risk_decomp['total_interaction_variance'] / p1_risk_decomp['portfolio_variance'] * 100
print(f"\n   Variance Decomposition:")
print(f"     Individual Asset Risk: {p1_individual_pct:.1f}%")
print(f"     Asset Interactions: {p1_interaction_pct:.1f}%")

# Portfolio 2 analysis
p2_risk_decomp = risk_decomposition_analysis(
    equal_weights, 
    p2_covariance_matrix.values, 
    portfolio2_returns.columns
)

print("\n\n🔵 PORTFOLIO 2 - DIVERSIFIED SECTORS:")
print(f"   Portfolio Volatility: {p2_risk_decomp['portfolio_volatility']:.2%}")
print("\n   Component Analysis:")
df2 = pd.DataFrame(p2_risk_decomp['component_analysis'])
print(df2[['Asset', 'Weight %', 'Risk Contrib %', 'Risk Ratio']].round(2).to_string(index=False))

# Variance decomposition
p2_individual_pct = p2_risk_decomp['total_individual_variance'] / p2_risk_decomp['portfolio_variance'] * 100
p2_interaction_pct = p2_risk_decomp['total_interaction_variance'] / p2_risk_decomp['portfolio_variance'] * 100
print(f"\n   Variance Decomposition:")
print(f"     Individual Asset Risk: {p2_individual_pct:.1f}%")
print(f"     Asset Interactions: {p2_interaction_pct:.1f}%")

### Visualize Risk Decomposition

Compare the variance decomposition between the two portfolios.

In [ ]:
# Visualization of risk decomposition comparison
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Variance Decomposition - Portfolio 1
variance_components_p1 = [p1_individual_pct, p1_interaction_pct]
labels_p1 = [f'Individual\nAsset Risk\n{p1_individual_pct:.1f}%', 
             f'Asset\nInteractions\n{p1_interaction_pct:.1f}%']
colors_pie = ['lightblue', 'lightcoral']

wedges1, texts1, autotexts1 = ax1.pie(variance_components_p1, labels=labels_p1,
                                       colors=colors_pie, autopct='',
                                       startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax1.set_title('Portfolio 1: Tech Sector\nVariance Decomposition', fontsize=14, fontweight='bold')

# Plot 2: Variance Decomposition - Portfolio 2
variance_components_p2 = [p2_individual_pct, p2_interaction_pct]
labels_p2 = [f'Individual\nAsset Risk\n{p2_individual_pct:.1f}%', 
             f'Asset\nInteractions\n{p2_interaction_pct:.1f}%']

wedges2, texts2, autotexts2 = ax2.pie(variance_components_p2, labels=labels_p2,
                                       colors=colors_pie, autopct='',
                                       startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax2.set_title('Portfolio 2: Diversified Sectors\nVariance Decomposition', fontsize=14, fontweight='bold')

# Plot 3: Risk Contribution by Asset - Portfolio 1
assets_p1 = df1['Asset'].values
risk_contrib_p1 = df1['Risk Contrib %'].values

bars1 = ax3.bar(range(len(assets_p1)), risk_contrib_p1, color='red', alpha=0.7, edgecolor='darkred', linewidth=2)
ax3.axhline(y=20, color='black', linestyle='--', linewidth=2, label='Equal Weight (20%)')
ax3.set_xticks(range(len(assets_p1)))
ax3.set_xticklabels(assets_p1, rotation=0, fontsize=11)
ax3.set_ylabel('Risk Contribution (%)', fontsize=12, fontweight='bold')
ax3.set_title('Portfolio 1: Tech Sector\nRisk Contribution by Asset', fontsize=14, fontweight='bold')
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

# Plot 4: Risk Contribution by Asset - Portfolio 2
assets_p2 = df2['Asset'].values
risk_contrib_p2 = df2['Risk Contrib %'].values

bars2 = ax4.bar(range(len(assets_p2)), risk_contrib_p2, color='blue', alpha=0.7, edgecolor='darkblue', linewidth=2)
ax4.axhline(y=20, color='black', linestyle='--', linewidth=2, label='Equal Weight (20%)')
ax4.set_xticks(range(len(assets_p2)))
ax4.set_xticklabels(assets_p2, rotation=0, fontsize=11)
ax4.set_ylabel('Risk Contribution (%)', fontsize=12, fontweight='bold')
ax4.set_title('Portfolio 2: Diversified Sectors\nRisk Contribution by Asset', fontsize=14, fontweight='bold')
ax4.legend(fontsize=11)
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars2:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Critical Insights from Risk Decomposition:")
print("="*70)
print(f"\n1. Portfolio Volatility Comparison:")
print(f"   Tech Sector: {p1_risk_decomp['portfolio_volatility']:.2%}")
print(f"   Diversified: {p2_risk_decomp['portfolio_volatility']:.2%}")
vol_reduction = (1 - p2_risk_decomp['portfolio_volatility']/p1_risk_decomp['portfolio_volatility']) * 100
print(f"   🎯 Risk Reduction: {vol_reduction:.1f}% through sector diversification!")

print(f"\n2. Interaction Variance (Correlation Effects):")
print(f"   Tech Sector: {p1_interaction_pct:.1f}% of total variance")
print(f"   Diversified: {p2_interaction_pct:.1f}% of total variance")
print(f"   ⚠️  High correlations in tech sector amplify portfolio risk!")
print(f"   ✅ Lower correlations across sectors reduce interaction risk!")

print(f"\n3. Key Takeaway:")
print(f"   Holding 5 tech stocks ≠ True diversification")
print(f"   Sector diversification provides REAL risk reduction")
print(f"   Same number of stocks, but {vol_reduction:.1f}% less risk!")

---

## Exercise 4: Diversification Benefit Calculation

Calculate the diversification benefit for both portfolios using equal weights:

$$\text{Diversification Benefit} = \frac{\text{Weighted Avg Volatility} - \text{Portfolio Volatility}}{\text{Weighted Avg Volatility}} \times 100\%$$

Compare the diversification benefits between the two portfolios.

In [ ]:
# Calculate diversification benefits
print("\n📊 DIVERSIFICATION BENEFIT ANALYSIS")
print("="*70)

# Portfolio 1: Tech Sector
p1_weighted_avg_vol = np.dot(equal_weights, p1_annual_volatilities)
p1_portfolio_vol = p1_risk_decomp['portfolio_volatility']
p1_div_benefit = (p1_weighted_avg_vol - p1_portfolio_vol) / p1_weighted_avg_vol * 100

print("\n🔴 Portfolio 1 - Tech Sector:")
print(f"   Weighted Average Volatility: {p1_weighted_avg_vol:.2%}")
print(f"   Actual Portfolio Volatility: {p1_portfolio_vol:.2%}")
print(f"   Diversification Benefit: {p1_div_benefit:.2f}%")

# Portfolio 2: Diversified
p2_weighted_avg_vol = np.dot(equal_weights, p2_annual_volatilities)
p2_portfolio_vol = p2_risk_decomp['portfolio_volatility']
p2_div_benefit = (p2_weighted_avg_vol - p2_portfolio_vol) / p2_weighted_avg_vol * 100

print("\n🔵 Portfolio 2 - Diversified Sectors:")
print(f"   Weighted Average Volatility: {p2_weighted_avg_vol:.2%}")
print(f"   Actual Portfolio Volatility: {p2_portfolio_vol:.2%}")
print(f"   Diversification Benefit: {p2_div_benefit:.2f}%")

print("\n" + "="*70)
print(f"\n✅ Diversification Benefit Comparison:")
print(f"   Tech Sector: {p1_div_benefit:.2f}% risk reduction")
print(f"   Diversified: {p2_div_benefit:.2f}% risk reduction")
benefit_diff = p2_div_benefit - p1_div_benefit
print(f"\n   🎯 Sector diversification provides {benefit_diff:.2f} percentage points")
print(f"      MORE diversification benefit than within-sector diversification!")

# Visualize diversification benefits
fig, ax = plt.subplots(figsize=(10, 6))

portfolios = ['Tech Sector', 'Diversified Sectors']
benefits = [p1_div_benefit, p2_div_benefit]
colors_bar = ['red', 'blue']

bars = ax.bar(portfolios, benefits, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels
for bar, benefit in zip(bars, benefits):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
           f'{benefit:.2f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Diversification Benefit (%)', fontsize=13, fontweight='bold')
ax.set_title('Diversification Benefit Comparison\n(Higher is Better)', 
            fontsize=15, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(benefits) * 1.2)

plt.tight_layout()
plt.show()

---

## Summary and Key Takeaways

### 🎯 What We Learned:

1. **Sector Concentration Risk**:
   - Stocks within the same sector are highly correlated (ρ ≈ 0.7-0.9)
   - High correlations limit diversification benefits
   - "5 tech stocks ≠ true diversification"

2. **Efficient Frontier Analysis**:
   - Diversified portfolios dominate single-sector portfolios
   - For the same return target, diversified portfolios achieve lower risk
   - The efficient frontier "shifts left" with better diversification

3. **Risk Decomposition Insights**:
   - Tech sector portfolio has higher interaction variance (correlation effects)
   - Diversified portfolio has more balanced risk contributions
   - Lower correlations mean less amplification of individual risks

4. **Practical Implications**:
   - Always diversify across sectors, not just within sectors
   - Monitor correlation structures in your portfolio
   - Sector ETFs can help achieve instant diversification
   - Risk reduction through diversification is "the only free lunch in finance"

### 📈 Real-World Applications:

- **Portfolio Construction**: Build multi-sector portfolios for better risk-adjusted returns
- **Risk Management**: Identify and reduce sector concentration risk
- **Asset Allocation**: Allocate across sectors based on correlation and risk contribution analysis
- **Performance Evaluation**: Compare portfolio efficiency using efficient frontier analysis

### 🔗 Connection to Investment Practice:

This lab demonstrates a fundamental principle of Modern Portfolio Theory:

> **Diversification works best when assets are not perfectly correlated!**

In practice:
- Individual investors: Use sector-diversified mutual funds or ETFs
- Portfolio managers: Monitor sector exposures and correlation changes
- Risk managers: Set sector concentration limits
- Financial advisors: Educate clients about true diversification

---

### 🎓 Assignment Complete!

**Key Formula to Remember**:

$$\sigma_p^2 = \sum_{i=1}^{n} \sum_{j=1}^{n} w_i w_j \sigma_i \sigma_j \rho_{ij}$$

When $\rho_{ij}$ is high (same sector), portfolio risk remains high.

When $\rho_{ij}$ is lower (different sectors), portfolio risk is reduced!

**Next Steps**: Apply these concepts to your own portfolio. Check your sector exposures and correlation structure. Are you truly diversified?